In [0]:
# 1. Core DataFrame/SQL transformations (covered already)
#
# select, filter, withColumn, groupBy, orderBy, distinct, drop

In [0]:
from pyspark.sql import SparkSession, Row

spark = SparkSession.builder.appName('Revision').getOrCreate()

In [0]:
# from pyspark.sql import SparkSession,Row

# spark = SparkSession.builder.appName('Revision').getOrCreate()



data = [
    Row(emp_id=1, name="Aarav",   department="Engineering", salary=95000, hire_date="2021-03-15"),
    Row(emp_id=2, name="Diya",    department="Engineering", salary=95000, hire_date="2020-06-01"),
    Row(emp_id=3, name="Kabir",   department="Engineering", salary=88000, hire_date="2022-01-10"),
    Row(emp_id=4, name="Meera",   department="Engineering", salary=76000, hire_date="2023-05-20"),
    Row(emp_id=5, name="Rohan",   department="Sales",       salary=72000, hire_date="2021-11-11"),
    Row(emp_id=6, name="Sanya",   department="Sales",       salary=72000, hire_date="2022-02-14"),
    Row(emp_id=7, name="Vihaan",  department="Sales",       salary=65000, hire_date="2020-09-09"),
    Row(emp_id=8, name="Ishita",  department="Marketing",   salary=60000, hire_date="2021-07-07"),
    Row(emp_id=9, name="Arjun",   department="Marketing",   salary=60000, hire_date="2023-01-01"),
]


df = spark.createDataFrame(data)

display(df)

In [0]:
df.show()

In [0]:
from pyspark.sql.functions import col
df.select(
    col("emp_id"),
    "name",
    "department",
    "salary",
    "hire_date"
).show()

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Step 1: define the window - this maps directly to your SQL's OVER(PARTITION BY ... ORDER BY ...)
windowSpec = Window.partitionBy("department").orderBy(F.desc("salary"))

# Step 2: add the dense_rank column - this maps to DENSE_RANK() in your SELECT
ranked_df = df.withColumn("salary_rank", F.dense_rank().over(windowSpec))

ranked_df.show()

# Step 3: filter - this is your WHERE/HAVING equivalent
result = ranked_df.filter(F.col("salary_rank") == 2)

result.show()




In [0]:
# %sql
# select * from (
# select *, dense_rank() over (paritition by deparment order by salary desc) as rn)
# where rn = 2;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,desc,dense_rank


windowspec = Window.partitionBy('department').orderBy(desc('salary'))


ranked_df = df.withColumn('salary_rank', dense_rank().over(windowspec))


ranked_df.show()

result = ranked_df.filter(col('salary_rank') == 2)

result.show()




In [0]:
display(df)

In [0]:
# df.filter(col('department') == 'Marketing').show()

In [0]:
from pyspark.sql.functions import when
df.withColumn('salary_band', when(col('salary') > 80000,'High').otherwise('Low')).show()

In [0]:
spark.range(3).show()


type(spark.range(3))

In [0]:
from pyspark.sql.functions import call_function, col
from pyspark.sql.types import IntegerType

spark.udf.register('myfunc', lambda x: x**2, IntegerType())

df.select(col('salary'), call_function('myfunc', col('salary')).alias('salary_squared')).show()

In [0]:
from pyspark.sql.functions import column,col
df.select(col('salary')).show()


df.select(column('salary')).show()

In [0]:
from pyspark.sql.functions import lit

df.withColumn('isRich',lit(False)).show()

In [0]:
from pyspark.sql.functions import length

df.withColumn('length',length(col('department'))).show()

In [0]:
from pyspark.sql.functions import expr,length

# df.select("*",expr("length(name)")).show()

df.select('*',
          length(col('name')).alias('character_length')).show()

In [0]:
df.where(col('department')=='Engineering').show()

In [0]:
df.withColumn('bonus',expr("salary * length(name)")).show()